# Step 5: SQL Revenue Analysis by Segment

We load our cleaned data and RFM segments into a SQLite database, then run SQL queries to analyze revenue patterns per customer segment.

In [ ]:
import pandas as pd
import sqlite3

# Load the two datasets
df      = pd.read_csv('../data/online_retail_clean.csv', parse_dates=['InvoiceDate'])
rfm     = pd.read_csv('../output/rfm_segments.csv')

# Create an in-memory SQLite database
conn = sqlite3.connect('../output/retail.db')

# Write both tables into the database
df.to_sql('transactions', conn, if_exists='replace', index=False)
rfm.to_sql('segments',    conn, if_exists='replace', index=False)

print('Database ready.')
print('transactions rows:', pd.read_sql('SELECT COUNT(*) FROM transactions', conn).iloc[0,0])
print('segments rows:    ', pd.read_sql('SELECT COUNT(*) FROM segments',    conn).iloc[0,0])

---
## Query 1: Total Revenue and Customer Count per Segment

In [ ]:
q1 = """
SELECT
    s.Segment,
    COUNT(DISTINCT s."Customer ID")          AS customers,
    ROUND(SUM(t.Quantity * t.Price), 2)      AS total_revenue,
    ROUND(AVG(t.Quantity * t.Price), 2)      AS avg_order_line_value
FROM transactions t
JOIN segments s ON t."Customer ID" = s."Customer ID"
GROUP BY s.Segment
ORDER BY total_revenue DESC
"""
q1_result = pd.read_sql(q1, conn)
q1_result

## Query 2: Revenue Share — What % of Total Revenue Does Each Segment Drive?

In [ ]:
q2 = """
WITH segment_revenue AS (
    SELECT
        s.Segment,
        ROUND(SUM(t.Quantity * t.Price), 2) AS total_revenue
    FROM transactions t
    JOIN segments s ON t."Customer ID" = s."Customer ID"
    GROUP BY s.Segment
),
grand_total AS (
    SELECT SUM(total_revenue) AS grand FROM segment_revenue
)
SELECT
    sr.Segment,
    sr.total_revenue,
    ROUND(sr.total_revenue * 100.0 / gt.grand, 1) AS revenue_pct
FROM segment_revenue sr, grand_total gt
ORDER BY sr.total_revenue DESC
"""
q2_result = pd.read_sql(q2, conn)
q2_result

## Query 3: Average Orders and Revenue Per Customer by Segment

In [ ]:
q3 = """
SELECT
    s.Segment,
    COUNT(DISTINCT t.Invoice)                                      AS total_orders,
    COUNT(DISTINCT s."Customer ID")                                AS customers,
    ROUND(COUNT(DISTINCT t.Invoice) * 1.0 /
          COUNT(DISTINCT s."Customer ID"), 1)                      AS avg_orders_per_customer,
    ROUND(SUM(t.Quantity * t.Price) /
          COUNT(DISTINCT s."Customer ID"), 2)                      AS revenue_per_customer
FROM transactions t
JOIN segments s ON t."Customer ID" = s."Customer ID"
GROUP BY s.Segment
ORDER BY revenue_per_customer DESC
"""
q3_result = pd.read_sql(q3, conn)
q3_result

## Query 4: Monthly Revenue Trend per Segment

In [ ]:
q4 = """
SELECT
    STRFTIME('%Y-%m', t.InvoiceDate)         AS month,
    s.Segment,
    ROUND(SUM(t.Quantity * t.Price), 2)      AS monthly_revenue
FROM transactions t
JOIN segments s ON t."Customer ID" = s."Customer ID"
GROUP BY month, s.Segment
ORDER BY month, s.Segment
"""
q4_result = pd.read_sql(q4, conn)
print(q4_result.shape)
q4_result.head(12)

## Query 5: Top 10 Customers by Revenue (with Segment)

In [ ]:
q5 = """
SELECT
    s."Customer ID",
    s.Segment,
    s.Recency,
    s.Frequency,
    ROUND(s.Monetary, 2)                     AS total_spent
FROM segments s
ORDER BY s.Monetary DESC
LIMIT 10
"""
q5_result = pd.read_sql(q5, conn)
q5_result

---
## Save All Query Results for Tableau

In [ ]:
q1_result.to_csv('../output/revenue_by_segment.csv',       index=False)
q2_result.to_csv('../output/revenue_share.csv',            index=False)
q3_result.to_csv('../output/orders_per_customer.csv',      index=False)
q4_result.to_csv('../output/monthly_revenue_trend.csv',    index=False)
q5_result.to_csv('../output/top10_customers.csv',          index=False)

conn.close()
print('All files saved to output/')